In [ ]:

%load_ext ElasticNotebook

# Analysis of NYC Flight Trends in Python


In [ ]:
%%RecordEvent
import IPython
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib
import sklearn
import matplotlib.pyplot as plt
import time

In [ ]:
%%RecordEvent
%%time
### cell 0 ###

factor = 1
flights_df = pd.read_csv('/home/dias-benchmarks/new_notebooks/nyc-flight/nyc_flights.csv')
flights_df = flights_df.sample(frac=factor, random_state=0)

In [ ]:
%%RecordEvent
%%time
### cell 1 ###

flights_df.shape, flights_df.columns, flights_df.dtypes

In [ ]:
%%RecordEvent
%%time
### cell 2 ###

flights_df.dest.unique()
flights_df.head(10)

## How many flights were there from NYC airports to Seattle in 2013?

In [ ]:
%%RecordEvent
%%time
### cell 3 ###

flights_df['dest'][flights_df['dest'] == 'SEA'].value_counts()

--There were a total of 3923 flights from NYC to Seattle in 2013--

## How many airlines fly from NYC to Seattle?

In [ ]:
%%RecordEvent
%%time
### cell 4 ###

flights_df['carrier'][flights_df['dest'] == 'SEA'].value_counts()

There are a total of 5 airline carriers that fly from NYC to Seattle. The total number of flights for each of the airline carriers is as follows:
DL - 1213, UA- 1117, AS-714, B6-514, AA-365

## How many unique air planes fly from NYC to Seattle?

In [ ]:
%%RecordEvent
%%time
### cell 5 ###

len(flights_df['tailnum'][flights_df['dest'] == 'SEA'].unique())

There are a total of 936 unique air planes from NYC to Seattle.

## What is the average arrival delay for flights from NC to Seattle?

In [ ]:
%%RecordEvent
%%time
### cell 6 ###

flights_df['arr_delay'][flights_df['dest'] == 'SEA'].mean()

The average arrival delay for flights from NYC to Seattle is -1.099. The negative value implies that instead of a delay, flights arrive before the scheduled time by 1.099 units.

## What proportion of flights to Seattle come from each NYC airport?

In [ ]:
%%RecordEvent
%%time
### cell 7 ###

f = flights_df[flights_df['dest'] == 'SEA'].groupby('origin').size()
f_total = len(flights_df[flights_df.dest == 'SEA'])
f.loc['EWR'] / f_total, f.loc['JFK'] / f_total

Total flights from EWR to Seattle: 1810
Proportion of flights from EWR to Seattle: 0.46
Total flights from JFK to Seattle: 2075
Proportion of flights from JFK to Seattle: 0.53

## Flights are often delayed. The following questions explore delay patterns:
## Which date has the largest average departure delay? Which date has the largest average arrival delay?

In [ ]:
%%RecordEvent
%%time
### cell 8 ###

df = flights_df.groupby(['month', 'day'], as_index=False).agg({'dep_delay': np.mean})
df2 = flights_df.groupby(['month', 'day'], as_index=False).agg({'arr_delay': np.mean})
df.loc[df['dep_delay'].idxmax()], df2.loc[df2['arr_delay'].idxmax()]

The date 8/3/2013 (8th March 2013) has the largest average departure delay equal to 83.53 units.
The date 8/3/2013 (8th March 2013) has the largest arrival delay, equal to 85.86 units.

## What was the worst day to fly out of NYC in 2013 if you dislike delayed flights?


In [ ]:
%%RecordEvent
%%time
### cell 9 ###

df

In [ ]:
%%RecordEvent
%%time
### cell 10 ###

df = flights_df.groupby(['day', 'month'], as_index=False).agg({'arr_delay': np.mean, 'dep_delay': np.mean})
df['total_delay'] = df['arr_delay'] + df['dep_delay']
df.sort_values('total_delay', ascending=False).head(1)

The worst day to fly out of NYC in 2013 was 8th March 2013. The average arrival delay was 85.86units, the average departure delay was 83.54 units and the total delay was 169 units.

## Are there any seasonal patterns in departure delays for flights from NYC?

In [ ]:
%%RecordEvent
%%time
### cell 11 ###

ds = flights_df.dropna(subset=['dep_delay']).groupby(['month'])['dep_delay'].mean()
ds

The average departure delay gradually increases from the month of January to July, followed by a decrease from August to November. Finally, there is an increase from December again. This signifies that certain seasonal factors may be responsible for decreasing departure delays in Fall and increasing departure delays in late Winter, Spring and Summer. Since the peak departure delays occur in Summer(June, July), other factors such as increased passenger traffic at airports (for holidaying) need to be explored for further analysis. 

## On average, how do departure delays vary over the course of a day?

In [ ]:
%%RecordEvent
%%time
### cell 12 ###

dt = flights_df.dropna(subset=['dep_delay']).groupby(['hour'])['dep_delay'].mean()
dt

There is a rapid increase in departure delays betwen 12 AM and 3 AM followed by a steep decline from 3 AM to 4 AM. Again, from 4 AM onwards, there is a gradual rise till 9 PM, forming a small peak around 11 PM. Finally, the departure delay starts decreasing from 11 PM to 12 AM. This may be attributed to the assumption that there are fewer flights at the timeslot of 11 PM to 12 AM which causes the rapid decline in departure delays. 

## Which flight departing NYC in 2013 flew the fastest?

In [ ]:
%%RecordEvent
%%time
### cell 13 ###

df = flights_df
df['speed'] = df['distance'] / df['air_time']
df[df['speed'] == df.speed.max()]

The flight 1499 with tailnum N666DN belonging to carrier DL travelling from LGA to ATL has the highest speed. 
It covers a distance of 762 miles in 65 minutes and has a speed of 11.723 mph.

## Which flights (i.e. carrier + flight + dest) happen every day? Where do they fly to?

In [ ]:
%%RecordEvent
%%time
### cell 14 ###

count = len(flights_df)
df = flights_df.groupby(['carrier', 'flight', 'dest']).size().reset_index(name='Size')
for i in df.index:
    if df.loc[i]['Size'] == 365:
        print('Carrier: %s, Flight: %s, Destination: %s' % (df.loc[i]['carrier'], df.loc[i]['flight'], df.loc[i]['dest']))

In [ ]:
%%RecordEvent
%%time
### cell 15 ###

gdf = flights_df
counts = gdf.groupby(['carrier', 'flight', 'dest']).size().reset_index(name='size')
complete_year = counts[counts['size'] == 365]
complete_year[['carrier', 'flight', 'dest']]

All the above 18 flights fly out of NYC every day.

## Which airline carrier has the best service in terms of the lowest average flight arrival and departure delays in June 2013?

In [ ]:
%%RecordEvent
%%time
### cell 16 ###

df = flights_df[flights_df['month'] == 6]
df = df.groupby('carrier', as_index=False).agg({'arr_delay': np.mean, 'dep_delay': np.mean})
df['total_delay'] = df['arr_delay'] + df['dep_delay']
for i in df.index:
    if df.loc[i]['total_delay'] == df['total_delay'].min():
        print(df.loc[i]['carrier'], df.loc[i]['total_delay'])
df

The carrier Hawaiian Airlines(HA) has the best service in terms of lowest average arrival and departure delays in June 2013. The lowest total delay is 3.3 units. The carrier with the highest average total delay (arrival delay + departure delay) is OO with an average total delay of 129.5 units. Thus, we can classify SkyWest Airlines (OO) as the worst carrier in terms of average total delay.

From the plot, we can see that the total delay (arrival delay + departure delay) for Hawaiian Airlines (HA) is the lowest. The median and quartile range for United Airlines is the lowest among all the other carriers.  The median and quartile range for Skywest Airlines(OO) is the highest from each of the plots. 

## What weather conditions are associated with flight delays leaving NYC? Use graphics to explore.

In [ ]:
%%RecordEvent
%%time
### cell 17 ###

weather_df = pd.read_csv('/home/dias-benchmarks/new_notebooks/nyc-flight/nyc_weather.csv')
df = flights_df
df_c = pd.merge(df, weather_df, on=['month', 'day', 'hour', 'origin'])
df_c.head(10)

After merging the flights and weather dataframes based on the columns month, day, hour and origin, two separate graphs have been plotted:

1. Departure Delay Vs. Humidity:
The Departure delay and Humidity plot indicates that there is a rise in departure delay as humidity increases till Humidity reaches the 50th unit. The departure delay reduces when humidity is between roughly 55-65 units. The graph is more or less constant except for the dip at the end. There are a lot of outliers. This signifies that there can be significantly high departure delays even when the humidity is not that high or low. This signifies that other factors such as wind speed, temperature, precipitation may be responsible for the numerous outliers. 

2. Arrival Delay Vs. Temperature:
The Arrival Delay and Temperature plot also indicates that there is a rise in arrival delay as temperature increases till roughly 35 units. There is a huge spike in Arrival delay at the 24-25th unit followed by the biggest spike in the 40th unit. The graph then remains more or less constant execpt for the dip at the end and a few spikes at the 45-55th unit, 64th, 85th and 9th units. Due to the significant dip in arrival delay at very high temperatures (90 units and above), we cannot conclude that there is a correlation between arrival delay and temperature. However, other factors may be responsible for the fluctuating arrival delay and the various spikes seen in the graph that we probably have not accounted for. Other factors not related to weather, such as waiting time at airports, passenger traffic, immigration queues etc. might have a significant effect on arrival delay which is beyond the purview of our dataset.